# 1 · Deine erste Markov-Kette

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/DE/jupyter/markov-book-de/01_erste_markov_kette.ipynb)

Eine **Markov-Kette** baut eine Folge Schritt für Schritt auf. In jedem Schritt schaut sie nur auf die
letzten paar Symbole (ihr *Gedächtnis*) und wählt das nächste Symbol mit bestimmten Wahrscheinlichkeiten.

In diesem Kapitel gibt es noch keine Musik. Wir nehmen Aktionen und Buchstaben, damit du siehst, wie wenig
dafür nötig ist, und lernen die drei GreasyPidgin-Objekte kennen, mit denen wir den ganzen Tag arbeiten:

- `MarkovRule` — eine Zeile des Rezepts: *„nach diesem kommt eines von jenen, mit diesen Gewichten"*
- `MarkovRules` — eine Sammlung von Regeln
- `MarkovChain` — folgt den Regeln und erzeugt eine Folge

Führe die Zellen von oben nach unten mit **Umschalt + Enter** aus. Die erste Zelle installiert alles.

In [ ]:
# Einrichtung -- läuft von selbst (beim ersten Mal etwa 20 Sekunden)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # welche Version von GreasyPidgin verwendet wird (Git-Branch oder -Tag)
TUTORIAL_REF  = "main"   # welche Version der Hilfsdatei dieses Tutorials verwendet wird
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/DE/jupyter/markov-book-de/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

wh.set_language("de")    # deutsche Meldungen, Beschriftungen und Notennamen
print("Bereit!")

## Eine Kette von Hand: Hommage à Cage *Song Books*

Die Zustände einer Kette müssen keine Noten sein, es können auch **Handlungen** sein.

Jede Regel sagt, was auf einen Zustand folgen kann und wie wahrscheinlich jede Möglichkeit ist.
Die Gewichte werden automatisch normiert: `[6, 3, 1]` funktioniert genauso gut wie `[0.6, 0.3, 0.1]`.
Zuerst reiner Zufall: Jede Zeile hat dieselben Gewichte, also ist es egal, was gerade passiert ist.

In [ ]:
aktionen = ["singen", "sprechen", "pause", "geräusch", "aktion"]

# Jede Zeile hat dieselben Gewichte (in der Reihenfolge von `aktionen`): reiner Zufall
zufall = MarkovChain(rules=MarkovRules([
    MarkovRule(a, aktionen, [3, 3, 2, 2, 1]) for a in aktionen
]))

Jetzt lassen wir die Kette 12 Schritte laufen. `seed` ist der *Startzustand* der Kette (kein Zufalls-Seed),
`n` ist die Zahl der neuen Schritte. Führe die Zelle mehrmals aus: jedes Mal kommt eine andere Folge heraus.

In [ ]:
print(" | ".join(zufall.getNElements(seed="pause", n=12)))

Jetzt dieselben Aktionen **mit Gedächtnis**: Was gerade geschah, verändert die Chancen für das Nächste.
Das ist der Unterschied zwischen einer Zufallsoperation und einer Markov-Kette.
Die Gewichte stehen wieder in der Reihenfolge `singen, sprechen, pause, geräusch, aktion`.

In [ ]:
gedaechtnis = MarkovChain(rules=MarkovRules([
    MarkovRule("singen",   aktionen, [6, 1, 1, 1, 1]),   # Gesang hält sich
    MarkovRule("sprechen", aktionen, [1, 4, 2, 1, 2]),
    MarkovRule("pause",    aktionen, [2, 2, 1, 1, 4]),   # nach der Stille passiert oft etwas
    MarkovRule("geräusch", aktionen, [1, 1, 3, 4, 1]),   # ein Geräusch bleibt oder verstummt
    MarkovRule("aktion",   aktionen, [1, 3, 4, 1, 1]),   # nach einer Aktion kehrt Ruhe ein
]))
print(gedaechtnis.rules)
print(" | ".join(gedaechtnis.getNElements(seed="pause", n=12)))

### Die Regeln sehen

Eine Tabelle mit Wahrscheinlichkeiten ist als Bild leichter zu lesen. Jede Zeile ist der *aktuelle* Zustand,
jede Spalte der *nächste*; je heller, desto wahrscheinlicher.

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(11, 4))
wh.plot_transition_matrix(zufall, label=str, ax=achsen[0])
wh.plot_transition_matrix(gedaechtnis, label=str, ax=achsen[1])
achsen[0].set_title("reiner Zufall")
achsen[1].set_title("mit Gedächtnis")
plt.tight_layout()
plt.show()

**Probier es aus:** Ändere die Gewichte in `gedaechtnis` so, dass auf eine Pause fast immer eine Aktion folgt.
Führe die Folge dann als Partitur aus: Eine Person liest sie vor, die Gruppe singt, spricht, schweigt,
macht Geräusche und Aktionen.

## Eine Kette aus Daten lernen

Regeln von Hand zu schreiben wird schnell mühsam. Eine Kette kann ihre Regeln auch **aus einer vorhandenen Folge
lernen**, indem sie zählt, was worauf folgte. Hier ist die Folge ein Wort, ein Buchstabe pro Schritt.

In [ ]:
wort = list("abracadabra")

gelernt = MarkovChain()
gelernt.generateRulesFromList(wort, order=1)   # Ordnung 1: nur der letzte Buchstabe wird gemerkt
print(gelernt.rules)

In [ ]:
neues_wort = gelernt.getNElements(seed="a", n=25)
print("".join(neues_wort))

Nach einem `a` geht das Originalwort mit `b`, `c`, `d` weiter oder (ganz am Ende) springt es zurück zum `a`.
Mehr weiß die Kette nicht — also erfindet sie neue Wörter mit demselben „Geschmack".

### Gedächtnis: Ordnung 2

Bei `order=2` merkt sich die Kette die letzten **zwei** Buchstaben. Dann braucht auch der Startwert zwei Buchstaben.

In [ ]:
gelernt2 = MarkovChain()
gelernt2.generateRulesFromList(wort, order=2)

print("".join(gelernt2.getNElements(seed=("a", "b"), n=25)))

**Probier es aus:**

- Ersetze `"abracadabra"` durch einen eigenen Satz, z. B. `list("der schnelle braune fuchs")`.
- Vergleiche `order=1`, `2`, `3`. Was passiert mit dem Ergebnis, wenn die Ordnung groß wird?
  (Tipp: Mit viel Gedächtnis und wenig Daten hat die Kette bei jedem Schritt nur eine Möglichkeit —
  sie kann nur das Original kopieren.)

Dieser Zielkonflikt zwischen *Kopieren* und *Erfinden* ist die Hauptidee dieses Workshops.
Im nächsten Kapitel machen wir dasselbe mit Noten aus einer MIDI-Datei.